In [1]:
import json, yaml, os
from pathlib import Path
from pipeline.config_adapter import adapt
from pipeline.normalizer import normalize_products
from pipeline.exporter import export_csv

# --- CONFIG ---
BRAND = "tissot"
TIMESTAMP = "20260311_180903" # <--- Verify this matches your folder name
RUN_FOLDER = Path(f"outputs/{BRAND}/listing/{TIMESTAMP}")

try:
    print(f"🚀 Resuming pipeline for {BRAND}...")

    # 1. Load Config
    with open(f"config/{BRAND}.yaml", "r", encoding="utf-8") as f:
        config = adapt(yaml.safe_load(f))
    print("✅ Config loaded.")

    # 2. Load Scraped Data (Step 4 Checkpoint)
    with open(RUN_FOLDER / "products_raw.json", "r", encoding="utf-8") as f:
        raw_products = json.load(f)
    print(f"✅ Loaded {len(raw_products)} raw scraped products.")

    # 3. Load Gender Mapping (Step 2 Checkpoint)
    url_data_map = {}
    coll_path = RUN_FOLDER / "url_collection.json"
    if coll_path.exists():
        with open(coll_path, "r", encoding="utf-8") as f:
            coll_data = json.load(f)
            url_data_map = {item["url"]: item for item in coll_data.get("urls", [])}
    print(f"✅ Loaded {len(url_data_map)} gender mappings.")

    # 4. Normalize (Step 5 - This uses the fixed code)
    image_map = {p["source_url"]: p.get("images", []) for p in raw_products}
    print("🛠️ Normalizing data...")
    normalized = normalize_products(raw_products, config, image_map, BRAND, url_data_map)
    print(f"✅ Normalized {len(normalized)} products.")

    # 5. Export CSV (Step 6)
    csv_path = RUN_FOLDER / "fixed_tissot_shopify_export.csv"
    export_csv(normalized, str(csv_path))

    print("-" * 30)
    print(f"🎉 SUCCESS! Shopify CSV created at:")
    print(csv_path.absolute())
    print("-" * 30)
    
    # 6. Preview first row
    if normalized:
        print("\nPreview of first product Handle/Title:")
        print(f"Handle: {normalized[0]['handle']}")
        print(f"Title:  {normalized[0]['title']}")

except Exception as e:
    print(f"❌ ERROR: {e}")

🚀 Resuming pipeline for tissot...
✅ Config loaded.
✅ Loaded 46 raw scraped products.
✅ Loaded 434 gender mappings.
🛠️ Normalizing data...
✅ Normalized 46 products.
------------------------------
🎉 SUCCESS! Shopify CSV created at:
c:\Users\anshi\Swiftcommerce\swiftcommerce\shopify-pipeline\outputs\tissot\listing\20260311_180903\fixed_tissot_shopify_export.csv
------------------------------

Preview of first product Handle/Title:
Handle: tissot-t-race-motogp-2026-45mm
Title:  Tissot T-Race MotoGP 2026 45mm
